# Gemini Embedding 2 & Vector Search 2.0 Workshop 🔑

---

# [Part 1] Multimodal Embedding & Local Validation

본 파트에서는 Gemini Embedding 2 모델을 기반으로 멀티모달(비디오) 전처리, 고속 청킹, 로컬 임베딩 생성 및 코사인 유사도를 직접 비교 측정하고, 시간 궤적을 분석하여 로컬 환경에서 가상의 하이브리드 검색을 시뮬레이션해 봅니다.

*   **비디오 전처리 및 고속 청킹 (FFmpeg Muxer)**
*   **Dense 임베딩 생성 및 로컬 코사인 유사도 분석 (Media Player 시각 검증)**
*   **비디오 시간 흐름 궤적 시각화 (t-SNE Trajectory)**
*   **로컬 하이브리드 검색 시뮬레이션 (BM25 + Dense 결합)**


> [!WARNING]
> **패키지 설치 및 자동 커널 재시작 안내**
> * 본 노트북을 최초 실행하는 시점에 라이브러리 설치 셀 (아래 코드 셀)을 실행하면 실습에 필요한 라이브러리들을 자동으로 설치합니다.
> * 설치 완료 시, 업데이트된 수많은 과학용 바이너리 패키지(NumPy, SciPy 등)를 메모리에 안전하게 새로 로드하기 위해 🚨주피터 커널이 1회 자동으로 재시작🚨되도록 설계되어 있습니다.
> * 자동 재시작이 수행되면 노트북 우측 상단의 커널 상태가 초기화됩니다. 이는 패키지 불일치 에러를 원천 차단하기 위한 정상적인 흐름이므로, 당황하지 마시고 **커널 재시작이 끝난 후 처음 셀부터 순서대로** 이어서 실행해 주시기 바랍니다.


In [ ]:
# Install necessary libraries
%pip install -qU google-genai google-cloud-vectorsearch google-cloud-aiplatform google-cloud-discoveryengine Pillow opencv-python numpy scikit-learn seaborn ipywidgets pyOpenSSL

try:
    # Import libraries
    import os
    import time
    from google import genai
    from google.genai import types
    from google.cloud import aiplatform
    from PIL import Image
    import cv2
    import numpy as np
    import ipywidgets as widgets
    from IPython.display import display, Image as IPImage, Video as IPVideo
    from sklearn.manifold import TSNE  # 라이브러리 바이너리 매핑 호환성 체크용 임포트

    print("라이브러리가 성공적으로 임포트되었습니다.")
except ImportError as e:
    print(f"⚠️ 패키지 업데이트 감지 (바이너리 불일치): {e}")
    print("새로운 라이브러리를 메모리에 반영하기 위해 주피터 커널을 자동으로 재시작합니다...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)


### API 인증 설정 (Authentication)

Gemini API를 호출하려면 사용자 인증이 필요합니다. Google Colab 환경에서는 내장된 인증 모듈을 사용할 수 있으며, 로컬 개발 환경에서는 `GEMINI_API_KEY` 환경 변수를 설정하거나 적절한 Google Cloud Application Default Credentials (ADC)를 구성해야 합니다. \
\
**기록해둔 Gemini API Key를 입력해 주세요:** \
**🔗 만약 키가 없다면 Google AI Studio에서 Gemini API Key를 생성 후 입력해주세요:** [https://aistudio.google.com/](https://aistudio.google.com/)

In [ ]:
import os
import getpass

if "GEMINI_API_KEY" not in os.environ:
    api_key_input = getpass.getpass("Gemini API Key를 입력해 주세요: ")
    if api_key_input.strip():
        os.environ["GEMINI_API_KEY"] = api_key_input.strip()
        print("Gemini API Key가 성공적으로 주입되었습니다.")
    else:
        print("경고: API Key가 입력되지 않았습니다. API 호출 시 오류가 발생할 수 있습니다.")
else:
    print("기존 세션에 이미 설정되어 있는 Gemini API Key를 사용합니다.")

In [ ]:
# Initialize the GenAI client
from google.cloud import storage
from google.cloud import aiplatform
from google import genai
import google.auth
import os
import subprocess

def get_project_id():
    """Attempts to auto-detect the active Google Cloud Project ID."""
    try:
        _, project_id = google.auth.default()
        if project_id:
            return project_id
    except Exception:
        pass
        
    try:
        project_id = subprocess.check_output(["gcloud", "config", "get-value", "project"]).decode("utf-8").strip()
        if project_id:
            return project_id
    except Exception:
        pass
        
    return "YOUR_PROJECT_ID"

# Auto-detect Project ID
PROJECT_ID = get_project_id()

print(f"Google Cloud Project ID 설정 완료: {PROJECT_ID}")
LOCATION = "us-central1"

# 1. Source Bucket: Read-only bucket containing shared sample data
SOURCE_BUCKET = "ai-multimodal-data"

# 2. Output Bucket: Project bucket for uploading results
OUTPUT_BUCKET = f"{PROJECT_ID}-bucket"

def create_bucket_if_not_exists(bucket_name, project_id, location="us-central1"):
    """Checks if a bucket exists, and creates it if it does not."""
    if project_id == "YOUR_PROJECT_ID":
        print("버킷을 생성하려면 실제 PROJECT_ID를 설정해 주세요.")
        return None

    storage_client = storage.Client(project=project_id)

    try:
        bucket = storage_client.get_bucket(bucket_name)
        print(f"버킷 {bucket_name}이(가) 이미 존재합니다.")
        return bucket
    except Exception:
        print(f"버킷 {bucket_name}을(를) 찾을 수 없습니다. 생성을 시도합니다...")

    try:
        bucket = storage_client.bucket(bucket_name)
        bucket.storage_class = "STANDARD"
        new_bucket = storage_client.create_bucket(bucket, location=location)
        print(f"버킷 {new_bucket.name}이(가) {new_bucket.location} 리전에 성공적으로 생성되었습니다.")
        return new_bucket
    except Exception as e:
        print(f"버킷 생성 중 오류 발생: {e}")
        print("사용자 계정에 '저장소 관리자(Storage Admin)' 역할이 부여되어 있고 프로젝트 ID가 올바른지 확인해 주세요.")
        return None

# Create the output bucket
create_bucket_if_not_exists(OUTPUT_BUCKET, PROJECT_ID, location="us-central1")

try:
    # Read GEMINI_API_KEY from environment
    api_key_val = os.environ.get("GEMINI_API_KEY")

    # Primary Developer API client
    client = genai.Client(
        api_key=api_key_val,
        http_options={'api_version': 'v1beta'}
    )
    
    # Vertex AI client for regional operations
    client_vertex = genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=LOCATION
    )
    
    # Initialize Agent Platform SDK for Vector Search
    aiplatform.init(project=PROJECT_ID, location=LOCATION)
    print("GenAI 클라이언트와 Agent Platform SDK가 성공적으로 초기화되었습니다.")
except Exception as e:
    print(f"클라이언트 초기화 중 오류 발생: {e}")
    print("Google Cloud 플랫폼 인증이 완료되었는지 확인해 주세요.")

# Define the model ID
MODEL_ID = 'gemini-embedding-2'
print(f"사용 모델: {MODEL_ID}")


## Cloud Storage 데이터 로드 및 확인

이 섹션에서는 GCS(Cloud Storage) 버킷에 연결하여 인증 상태를 검증하고 검색 엔진 색인(Indexing)에 사용할 실습용 이미지 및 비디오 파일 목록과 샘플 프레임을 확인합니다.


In [ ]:
# Check Authentication and Preview Data from GCS
from google.cloud import storage
from PIL import Image
import io
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, HTML
import os
import traceback
import base64

print(f"소스 버킷 액세스 확인 중: {SOURCE_BUCKET}...")
try:
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(SOURCE_BUCKET)

    blobs = list(bucket.list_blobs(max_results=10000))
    print(f"버킷에서 {len(blobs)}개의 파일 목록을 로드했습니다 (안전한 프리뷰를 위해 10000개로 제한).")

    print("\n샘플 확인:")

    images = []
    videos = []

    for blob in blobs:
        if blob.name.endswith(('.png', '.jpg', '.jpeg')):
            images.append(blob)
        elif blob.name.endswith('.mp4'):
            videos.append(blob)

    print(f"총 이미지 수: {len(images)}")
    print(f"총 비디오 수: {len(videos)}")

    # Display up to 3 images
    img_widgets = []
    print("\n이미지 프리뷰:")
    for blob in images[:3]:
        print(f"- {blob.name} ({blob.content_type})")
        image_bytes = blob.download_as_bytes()
        out = widgets.Output()
        with out:
            display(IPImage(data=image_bytes, width=200))
        img_widgets.append(out)

    if img_widgets:
        display(widgets.HBox(img_widgets))

    if videos:
        print(f"\n{len(videos)}개의 비디오를 발견했습니다. 대용량 파일 다운로드를 방지하기 위해 프리뷰를 생략합니다.")

    if not images and not videos:
        print("버킷에 이미지 또는 비디오 파일이 존재하지 않습니다.")

except Exception as e:
    print(f"버킷 액세스 중 오류 발생: {e}")
    traceback.print_exc()
    print("인증 자격 증명이 올바르고 버킷 접근 권한이 있는지 확인해 주세요.")


## 1단계: 비디오 전처리 및 세그먼트 분할 (Chunking)

대용량 비디오 파일(40~60분 분량)의 경우 전체 영상에 대한 단일 임베딩을 생성하면 정보 손실(Embedding Dilution)이 발생합니다. 특정 시점의 세부적인 시맨틱을 포착하기 위해 영상을 의미 있는 단위(예: 10초 세그먼트)로 잘게 쪼개는 분할 프로세스가 필수적입니다.

### 고성능 FFmpeg 비디오 청커 (Stream Copy)

본 실습에서는 고효율 스트림 복사 연산이 가능한 **FFmpeg** segmenter 방식을 기본으로 사용합니다:

*   **FFmpeg (Stream Copy)**: 비디오와 오디오 스트림을 재인코딩 없이 원본 해상도 그대로 초고속 복사하므로, 연산 시간이 0.1초 미만으로 단축되며 고품질 오디오 채널이 완벽하게 보존됩니다.
*   **libx264 소프트웨어 트랜스코딩 (Fallback)**: 스트림 복사 실패 시 CPU 소프트웨어 디바이스를 통해 코덱 변환 분할을 재수행해 호환성을 확보합니다.

본 실습에서는 Google Cloud x Team USA - Behind the tech 영상인 `gs://ai-multimodal-data/team_usa_tech.mp4` 경로의 비디오를 다운로드하여 진행합니다.

In [ ]:
import os
import cv2
import subprocess
from google.cloud import storage

VIDEO_URI = "gs://ai-multimodal-data/team_usa_tech.mp4"
OUTPUT_BUCKET = OUTPUT_BUCKET
OUTPUT_FOLDER = "vid-chunks"
CHUNK_DURATION = 10  # seconds

# Download the video file locally for processing
def download_blob(bucket_name, source_blob_name, destination_file_name):
    """Downloads a blob from the bucket."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(source_blob_name)
    blob.download_to_filename(destination_file_name)
    print(f"Blob {source_blob_name}이(가) {destination_file_name} 경로로 다운로드되었습니다.")

# Extract bucket and blob name dynamically from URI
if VIDEO_URI.startswith("gs://"):
    parts = VIDEO_URI[5:].split('/', 1)
    src_bucket_name = parts[0]
    src_blob_name = parts[1]
    local_video_path = os.path.basename(src_blob_name)
else:
    src_bucket_name = "ai-multimodal-data"
    src_blob_name = "team_usa_tech.mp4"
    local_video_path = "team_usa_tech.mp4"

# Download if not exists
if not os.path.exists(local_video_path):
    print(f"비디오 다운로드 중: {VIDEO_URI}...")
    try:
        download_blob(src_bucket_name, src_blob_name, local_video_path)
    except Exception as e:
        print(f"비디오 다운로드 실패: {e}")
        print("소스 버킷에 대한 접근 권한이 있는지 확인해 주세요.")
else:
    print(f"로컬 파일 {local_video_path}이(가) 이미 존재합니다.")

def chunk_video_opencv(video_path, chunk_duration_sec=10, output_dir="chunks"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # Clear directory of old chunks to prevent index issues
    for f in os.listdir(output_dir):
        if f.endswith(".mp4"):
            try:
                os.remove(os.path.join(output_dir, f))
            except Exception:
                pass

    # Resolve ffmpeg executable
    ffmpeg_bin = "ffmpeg"
    try:
        import imageio_ffmpeg as im_ffmpeg
        ffmpeg_bin = im_ffmpeg.get_ffmpeg_exe()
    except ImportError:
        print("imageio-ffmpeg 라이브러리가 설치되어 있지 않습니다. 자동 설치를 진행합니다...")
        try:
            import sys
            subprocess.run([sys.executable, "-m", "pip", "install", "imageio-ffmpeg"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            import imageio_ffmpeg as im_ffmpeg
            ffmpeg_bin = im_ffmpeg.get_ffmpeg_exe()
            print("imageio-ffmpeg 라이브러리가 성공적으로 설치되었습니다!")
        except Exception as install_err:
            print(f"imageio-ffmpeg 라이브러리 자동 설치 실패: {install_err}")
            print("인터넷 연결 상태를 확인하시거나 시스템에 ffmpeg를 수동으로 설치해 주세요.")

    print(f"고성능 FFmpeg 스트림 복사 방식을 사용하여 {video_path} 비디오 분할(Chunking)을 시작합니다...")
    
    # Using ffmpeg segment muxer
    chunk_pattern = os.path.join(output_dir, "chunk_%d.mp4")
    cmd = [
        ffmpeg_bin, "-y", "-i", video_path,
        "-c", "copy", 
        "-f", "segment",
        "-segment_time", str(chunk_duration_sec),
        "-reset_timestamps", "1",
        chunk_pattern
    ]
    
    try:
        try:
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            print("FFmpeg 복사 방식 비디오 분할이 완료되었습니다!")
        except Exception as e:
            print(f"FFmpeg 복사 방식 분할 실패. 소프트웨어 인코딩(libx264) 재인코딩 분할 방식을 시도합니다: {e}")
            cmd = [
                ffmpeg_bin, "-y", "-i", video_path,
                "-c:v", "libx264", "-c:a", "aac",
                "-f", "segment",
                "-segment_time", str(chunk_duration_sec),
                "-reset_timestamps", "1",
                chunk_pattern
            ]
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            print("FFmpeg 재인코딩 방식 비디오 분할이 성공적으로 완료되었습니다!")
    except FileNotFoundError:
        print("오류: 시스템에서 ffmpeg 실행 파일을 찾을 수 없습니다. Colab 환경인지 또는 ffmpeg가 설치되어 있는지 확인해 주세요.")
        return []
    except Exception as e:
        print(f"비디오 분할 작업 실행 중 오류 발생: {e}")
        return []

    # Gather metadata and calculate start/end times
    metadata = []
    chunk_files = sorted(
        [f for f in os.listdir(output_dir) if f.startswith("chunk_") and f.endswith(".mp4")],
        key=lambda x: int(x.split("_")[1].split(".")[0])
    )
    
    for i, chunk_name in enumerate(chunk_files):
        chunk_file = os.path.join(output_dir, chunk_name)
        
        # Get actual duration of each chunk using OpenCV
        cap = cv2.VideoCapture(chunk_file)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 and total_frames > 0 else chunk_duration_sec
        cap.release()
        
        start_time = i * chunk_duration_sec
        end_time = start_time + duration
        
        metadata.append({
            "chunk_id": i,
            "file_path": chunk_file,
            "start_time": start_time,
            "end_time": end_time
        })
        
    print(f"정밀 메타데이터를 포함한 {len(metadata)}개의 비디오 청크 세그먼트를 생성했습니다.")
    return metadata

# Run chunking
if os.path.exists(local_video_path):
    print("비디오 분할(Chunking) 전처리를 시작합니다...")
    chunk_metadata = chunk_video_opencv(local_video_path, CHUNK_DURATION, "local_chunks")
    print(f"{len(chunk_metadata)}개 청크에 대한 메타데이터 생성이 완료되었습니다.")
    print("메타데이터 샘플:", chunk_metadata[0] if chunk_metadata else "없음")
else:
    print("비디오 다운로드가 완료되지 않아 분할 전처리를 생략합니다.")

In [ ]:
def upload_directory(local_path, bucket_name, gcs_folder):
    """Uploads a directory to GCS."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    print(f"GCS 버킷(gs://{bucket_name}/{gcs_folder})으로 청크 파일 업로드를 시작합니다...")
    count = 0
    for root, dirs, files in os.walk(local_path):
        for file in files:
            local_file = os.path.join(root, file)
            remote_path = os.path.join(gcs_folder, file)
            blob = bucket.blob(remote_path)
            blob.upload_from_filename(local_file)
            count += 1

    print(f"총 {count}개의 파일 업로드를 완료했습니다.")

# Upload chunks if they were created
if 'chunk_metadata' in globals() and chunk_metadata:
    try:
        upload_directory("local_chunks", OUTPUT_BUCKET, OUTPUT_FOLDER)
    except Exception as e:
        print(f"GCS 업로드 실패: {e}")
        print("대상 GCS 버킷에 대한 쓰기 권한이 있는지 확인해 주세요.")
else:
    print("업로드할 청크 파일이 존재하지 않습니다.")

## 멀티모달 임베딩 생성 (Generating Embeddings)

이 섹션에서는 최신 `gemini-embedding-2` 모델을 호출하여 텍스트뿐만 아니라 비디오 등 다양한 미디어 데이터를 3072차원의 임베딩 벡터로 변환하는 범용 함수를 정의하고 실행합니다.


In [ ]:
def generate_multimodal_embedding(content_path, content_type):
    """Generates a multimodal embedding using Gemini Embedding 2 on Developer API."""
    try:
        import base64
        from google.genai import types
        
        # 1. Determine mime type
        if content_type == 'video':
            mime_type = 'video/mp4'
        elif content_type == 'audio':
            mime_type = 'audio/mpeg'
        else:
            mime_type = 'image/jpeg'
            
        # 2. Build contents parameter
        if content_type == 'text':
            contents = content_path
        else:
            # Check if GCS path
            if isinstance(content_path, str) and content_path.startswith("gs://"):
                from google.cloud import storage
                storage_client = storage.Client()
                parts = content_path[5:].split('/', 1)
                bucket_name = parts[0]
                blob_name = parts[1] if len(parts) > 1 else ""
                
                bucket = storage_client.bucket(bucket_name)
                blob = bucket.blob(blob_name)
                file_bytes = blob.download_as_bytes()
                
                contents = types.Part.from_bytes(
                    data=file_bytes,
                    mime_type=mime_type
                )
            else:
                # Local file
                with open(content_path, 'rb') as f:
                    file_bytes = f.read()
                contents = types.Part.from_bytes(
                    data=file_bytes,
                    mime_type=mime_type
                )
        
        # 3. Call Developer API client
        response = client.models.embed_content(
            model='gemini-embedding-2',
            contents=contents,
            config = types.EmbedContentConfig(
                output_dimensionality=3072,
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=10,
                        initial_delay=1.0,
                        max_delay=3.0
                    )
                )
            )
        )
        
        # Extract embedding values
        embedding_values = response.embeddings[0].values
        return embedding_values
        
    except Exception as e:
        print(f"임베딩 생성 실패 ({content_path}): {e}")
        return None

def display_test_previews(results, title="검색 매칭 결과"):
    """Displays a horizontal gallery of search results (images and video player widgets)."""
    import ipywidgets as widgets
    from IPython.display import display, Image as IPImage, HTML
    import base64
    import os

    print(f"\n🎬 [{title} - 비주얼 갤러리 프리뷰]:")
    
    outs = []
    # Display at most top 5 results in the preview row
    for i, res in enumerate(results[:5]):
        # Handle formats where item is nested or is the dictionary itself
        item = res.get('item', res)
        
        path = item.get('path', item.get('file_path', ''))
        if not path:
            continue
            
        media_type = item.get('type', 'video_chunk')
        item_id = str(item.get('id', item.get('chunk_id', '')))
        score = res.get('score', 0.0)
        
        o = widgets.Output()
        with o:
            print(f"Rank {i+1} (Score: {score:.3f})")
            print(f"ID: {item_id[:20]}...")
            
            try:
                media_bytes = None
                
                # Step 1: Try original path
                if path.startswith("gs://"):
                    try:
                        from google.cloud import storage
                        storage_client = storage.Client()
                        parts = path[5:].split('/', 1)
                        bucket = storage_client.bucket(parts[0])
                        blob = bucket.blob(parts[1])
                        media_bytes = blob.download_as_bytes()
                    except Exception:
                        pass
                else:
                    if os.path.exists(path):
                        with open(path, 'rb') as f:
                            media_bytes = f.read()

                # Step 2: Try local chunks folder
                if media_bytes is None and media_type in ['video', 'video_chunk']:
                    try:
                        # Extract chunk index dynamically depending on ID formats
                        if '_' in item_id:
                            chunk_idx = item_id.rsplit('_', 1)[1]
                        else:
                            chunk_idx = item_id
                        
                        local_fallback_path = f"local_chunks/chunk_{chunk_idx}.mp4"
                        if os.path.exists(local_fallback_path):
                            with open(local_fallback_path, 'rb') as f:
                                media_bytes = f.read()
                    except Exception:
                        pass

                # Step 3: Try GCS bucket folder
                if media_bytes is None and media_type in ['video', 'video_chunk']:
                    try:
                        if '_' in item_id:
                            parts = item_id.rsplit('_', 1)
                            video_filename = parts[0]
                            chunk_idx = parts[1]
                        else:
                            video_filename = "team_usa_tech.mp4"
                            chunk_idx = item_id
                            
                        from google.cloud import storage
                        storage_client = storage.Client()
                        bucket = storage_client.bucket("ai-multimodal-data")
                        blob_name = f"vid-chunks/{video_filename}_chunks/chunk_{chunk_idx}.mp4"
                        
                        blob = bucket.blob(blob_name)
                        if blob.exists():
                            media_bytes = blob.download_as_bytes()
                    except Exception:
                        pass
                
                if media_bytes:
                    if media_type == 'image':
                        display(IPImage(data=media_bytes, width=150))
                    elif media_type in ['video', 'video_chunk']:
                        encoded_video = base64.b64encode(media_bytes).decode('utf-8')
                        video_html = f'''
                        <div style="position: relative; z-index: 9999; pointer-events: auto !important; display: inline-block;">
                            <video width="150" controls  style="position: relative; z-index: 9999; pointer-events: auto !important; display: block;">
                                <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                            </video>
                        </div>
                        '''
                        display(HTML(video_html))
                else:
                    print("[미디어 파일 없음]")
            except Exception as e:
                print(f"[렌더링 실패: {e}]")
        outs.append(o)
        
    if outs:
        display(widgets.HBox(outs))
    else:
        print("표시할 수 있는 매칭 결과가 없습니다.")

print("Developer API 기반 멀티모달 임베딩 생성 및 갤러리 프리뷰 출력 헬퍼 함수 정의 완료")

In [ ]:
# Generate dense embeddings for the chunks
# Limit to the first LIMIT = 10 chunks for demonstration
LIMIT = 10
if 'chunk_metadata' in globals() and chunk_metadata:
    valid_chunks = chunk_metadata[:LIMIT]
    print(f"상위 {len(valid_chunks)}개 비디오 청크의 고차원 밀집 임베딩 벡터 생성을 시작합니다...")
    for item in valid_chunks:
        local_path = item['file_path']
        print(f"임베딩 벡터화 작업 중: {local_path}...")
        embedding = generate_multimodal_embedding(local_path, 'video')
        if embedding is not None:
            item['dense_embedding'] = embedding
        else:
            print(f"경고: {local_path} 임베딩 생성 실패")
    print(f"{len(valid_chunks)}개 비디오 청크에 대한 밀집 임베딩 생성이 완료되었습니다.")
else:
    print("청크 메타데이터가 존재하지 않습니다.")


### 📊 개발자 인터랙티브 분석: 임베딩 유사도 측정 및 시간 차원 비디오 궤적 (t-SNE Trajectory) 시각화

본격적으로 관리형 벡터 데이터베이스(Vector Search)에 데이터를 저장하기에 앞서, 로컬 메모리 상에서 직접 수학적 비교 분석을 수행하여 임베딩 공간의 구조와 시맨틱 분산에 대한 직관을 형성합니다:
1.  **임베딩 간 코사인 유사도 분석**: 넘파이(numpy) 기반의 수동 연산 함수를 구축하여 특정 레퍼런스 청크 대비 '가장 시맨틱이 유사한 장면'과 '가장 이질적인 장면'을 비교 추출합니다.
2.  **t-SNE 기반 고차원 시각화**: 3072차원에 달하는 고차원 임베딩 행렬을 인간이 직관적으로 파악할 수 있도록 2차원 공간으로 차원 축소(Dimensionality Reduction)를 수행합니다.
3.  **시간 흐름 별 비디오 리듬 분석**: 2차원 축소 도표 상의 포인트들을 시간 순서(Chronological order)에 따라 화살표 연결선으로 그려냄으로써 영상 속 씬 전환(Scene cuts)과 흐름의 동적 강도를 실시간으로 추적합니다.


In [ ]:
# ==========================================
# 1. Embedding Distance & Similarity Demo
# ==========================================
import numpy as np
import base64
import os
import cv2
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

def cosine_similarity(v1, v2):
    """Calculate the cosine similarity between two vectors using numpy.dot."""
    if v1 is None or v2 is None:
        return 0.0
    a = np.array(v1)
    b = np.array(v2)
    if a.size == 0 or b.size == 0:
        return 0.0
    if a.shape != b.shape:
        print(f"경고: 두 벡터의 차원이 일치하지 않습니다! {a.shape} 대 {b.shape}")
        return 0.0
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return np.dot(a, b) / (norm_a * norm_b)

def render_local_video(file_path, title, width=320):
    """Encodes a local video file to base64 and returns an HTML element to play it inline."""
    if not os.path.exists(file_path):
        return f"<div style='border: 1px solid #ddd; padding: 15px; border-radius: 8px; background-color: #fff;'><h4>{title}</h4><p style='color: red;'>파일을 찾을 수 없음: {file_path}</p></div>"
    try:
        return f"""
        <div style="max-width: 400px; margin: 10px auto; border: 1px solid #ddd; padding: 15px; border-radius: 8px; text-align: center; background-color: #fff; box-shadow: 0 4px 6px rgba(0,0,0,0.1); position: relative; z-index: 9999; pointer-events: auto !important;">
            <h4 style="margin-bottom: 10px; color: #333;">{title}</h4>
            <div style="display: inline-block; width: {width}px; max-width: 100%;">
                <video width="{width}" controls style="display: block; width: 100%; height: auto; margin: 0 auto; position: relative; z-index: 9999; pointer-events: auto !important;">
                    <source src="{file_path}" type="video/mp4">
                    Your browser does not support the video tag.
                </video>
            </div>
            <p style="font-size: 11px; color: #777; margin-top: 8px;">파일: {os.path.basename(file_path)}</p>
        </div>
        """
    except Exception as e:
        return f"<div style='border: 1px solid #ddd; padding: 15px; border-radius: 8px;'><h4>{title}</h4><p style='color: red;'>재생 실패: {e}</p></div>"


if 'valid_chunks' in globals() and len(valid_chunks) > 1:
    print("=== 로컬 임베딩 코사인 유사도 연산 분석 (DB 연결 없이 진행) ===")

    # Let's take the very first video chunk as our reference query
    ref_chunk = valid_chunks[0]
    ref_embedding = ref_chunk['dense_embedding']
    ref_label = f"Chunk 0 ({ref_chunk['start_time']}s - {ref_chunk['end_time']}s)"
    print(f"기준 세그먼트 (Reference): {ref_label}\n")

    similarities = []
    for i, item in enumerate(valid_chunks[1:], start=1):
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            sim = cosine_similarity(ref_embedding, item['dense_embedding'])
            similarities.append((i, item, sim))

    if not similarities:
        print("경고: 유효한 유사도를 계산할 수 없습니다. 청크들에 밀집 임베딩 벡터가 정상 생성되었는지 확인해 주세요.")
    else:
        # Sort by similarity
        similarities.sort(key=lambda x: x[2], reverse=True)

        # 1. Find the Most Similar Chunk
        most_similar_idx, most_similar_chunk, max_sim = similarities[0]
        print(f"✅ [가장 유사한 세그먼트 발견] -> 코사인 유사도 점수: {max_sim:.4f}")
        print(f"   시간대 범위: {most_similar_chunk['start_time']}초 - {most_similar_chunk['end_time']}초")
        print(f"   로컬 파일 경로: {most_similar_chunk['file_path']}")
        print(f"   결과 해석: 기준 세그먼트와 가장 근접한 비즈니스/시맨틱 맥락 및 시각적 피처를 공유하는 구간입니다.\n")

        # 2. Find the Most Distant (Furthest) Chunk
        least_similar_idx, least_similar_chunk, min_sim = similarities[-1]
        print(f"❌ [가장 이질적인 세그먼트 발견] -> 코사인 유사도 점수: {min_sim:.4f}")
        print(f"   시간대 범위: {least_similar_chunk['start_time']}초 - {least_similar_chunk['end_time']}초")
        print(f"   로컬 파일 경로: {least_similar_chunk['file_path']}")
        print(f"   결과 해석: 기준 세그먼트의 맥락적 주제와 시각적 씬이 가장 상이하고 거리가 먼 구간입니다.\n")

        # 🎬 인터랙티브 비디오 셀렉터 & 플레이어 구성
        print("🎬 [인터랙티브 비디오 플레이어] 아래 드롭다운에서 원하는 청크 세그먼트를 선택해 비디오 화면을 확인하세요:")
        
        dropdown_options = []
        for i, item in enumerate(valid_chunks):
            dropdown_options.append((f"Chunk {item['chunk_id']} ({item['start_time']}초 - {item['end_time']}초)", i))
            
        chunk_dropdown = widgets.Dropdown(
            options=dropdown_options,
            value=0, # Default to Chunk 0
            description='선택 세그먼트:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='350px')
        )
        
        video_output = widgets.Output(layout=widgets.Layout(margin='10px 0'))
        
        def on_dropdown_change(change):
            if change['type'] == 'change' and change['name'] == 'value':
                selected_idx = change['new']
                selected_chunk = valid_chunks[selected_idx]
                file_path = selected_chunk['file_path']
                
                # 유사도 찾기
                if selected_idx == 0:
                    sim_text = " (기준 세그먼트, 유사도: 1.0000)"
                else:
                    sim_val = next((x[2] for x in similarities if x[1]['chunk_id'] == selected_chunk['chunk_id']), 0.0)
                    sim_text = f" (코사인 유사도: {sim_val:.4f})"
                
                with video_output:
                    clear_output(wait=True)
                    html_code = render_local_video(file_path, f"Chunk {selected_chunk['chunk_id']}{sim_text}")
                    display(HTML(html_code))
                    
        chunk_dropdown.observe(on_dropdown_change, names='value')
        
        # 레이아웃 표시
        display(widgets.VBox([chunk_dropdown, video_output]))
        
        # 첫 구동 화면: Chunk 0 로드
        with video_output:
            clear_output(wait=True)
            html_code = render_local_video(ref_chunk['file_path'], "Chunk 0 (기준 세그먼트, 유사도: 1.0000)")
            display(HTML(html_code))
else:
    print("경고: 'valid_chunks'가 존재하며 적어도 2개 이상의 밀집 임베딩 적용 청크가 포함되어 있는지 확인해 주세요.")


In [ ]:
# ===================================================
# 2. t-SNE Dimensionality Reduction & Trajectory Plot
# ===================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import inspect

if 'valid_chunks' in globals() and len(valid_chunks) >= 3:
    print("비디오 청크 임베딩에 대한 t-SNE 차원 축소(Dimensionality Reduction)를 수행 중입니다...")

    # Gather embeddings and chronological metadata
    embeddings = []
    chronological_labels = []
    chunk_indices = []

    for i, item in enumerate(valid_chunks):
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            embeddings.append(item['dense_embedding'])
            chronological_labels.append(f"{item['start_time']}s - {item['end_time']}s")
            chunk_indices.append(i)

    if len(embeddings) < 3:
        print("경고: t-SNE 비디오 궤적(Trajectory) 시각화를 수행하려면 최소 3개 이상의 유효 밀집 임베딩 벡터가 필요합니다.")
    else:
        # Convert to numpy array
        embeddings_matrix = np.array(embeddings)

        # Run t-SNE (Set perplexity low for small dataset demo)
        perplexity_val = min(5, len(embeddings) - 1)
        
        # Dynamically support both older (n_iter) and newer (max_iter) scikit-learn versions
        tsne_params = inspect.signature(TSNE.__init__).parameters
        tsne_kwargs = {
            "n_components": 2,
            "perplexity": perplexity_val,
            "random_state": 42
        }
        if "max_iter" in tsne_params:
            tsne_kwargs["max_iter"] = 1000
        else:
            tsne_kwargs["n_iter"] = 1000
            
        tsne = TSNE(**tsne_kwargs)
        embeddings_2d = tsne.fit_transform(embeddings_matrix)

        # Plotting
        plt.figure(figsize=(10, 8))
        sns.set_theme(style="whitegrid")

        # 1. Plot the embedding coordinates
        scatter = plt.scatter(
            embeddings_2d[:, 0],
            embeddings_2d[:, 1],
            c=chunk_indices,
            cmap='viridis',
            s=150,
            zorder=3,
            edgecolors='black',
            linewidth=1.5
        )

        # 2. Draw the Video Trajectory (Connecting coordinates in chronological order)
        for i in range(len(embeddings_2d) - 1):
            plt.annotate(
                '',
                xy=(embeddings_2d[i+1, 0], embeddings_2d[i+1, 1]),
                xytext=(embeddings_2d[i, 0], embeddings_2d[i, 1]),
                arrowprops=dict(
                    arrowstyle="->",
                    color="red",
                    lw=1.5,
                    ls="--",
                    alpha=0.6,
                    connectionstyle="arc3,rad=0.1"
                )
            )

        # Annotate points with their timestamps
        for idx, (x, y) in enumerate(embeddings_2d):
            plt.text(
                x + 0.2,
                y + 0.2,
                f"Chunk {idx}\n({chronological_labels[idx]})",
                fontsize=9,
                weight='bold',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", edgecolor="gray", alpha=0.8),
                zorder=4
            )

        plt.colorbar(scatter, label="Chronological Chunk Index")
        plt.title("Chronological Video Embedding Trajectory (t-SNE Projections)", fontsize=14, weight='bold')
        plt.xlabel("t-SNE Dimension 1", fontsize=11)
        plt.ylabel("t-SNE Dimension 2", fontsize=11)
        plt.grid(True, which='both', linestyle=':', alpha=0.5)

        plt.tight_layout()
        plt.show()

        print("\n💡 비디오 궤적 시각화 그래프 해석 가이드:")
        print("- 군집된 포인트: 시간축 상 인접한 청크들이 서로 뭉쳐 있다면, 해당 구간 동안 카메라 구도나 비디오 장면 흐름이 상대적으로 정적이거나 연속적임을 뜻합니다.")
        print("- 큰 격차: 선 사이 간격이 벌어지며 크게 도약하는 구간은 장면 전환, 화면 페이드, 또는 급격한 카메라 동적 무브먼트가 발생했음을 지칭합니다.")
        print("- 루프/회귀: 궤적이 이전 좌표 영역으로 다시 돌아간다면 영상 연출 상의 동일 씬 반복(예: 다큐멘터리 아나운서 스튜디오 화면으로의 복귀)을 암시합니다.")
else:
    print("경고: t-SNE 궤적 그래프를 생성하려면 valid_chunks에 최소 3개 이상의 데이터가 들어있는지 확인해 주세요.")


### 🔍 밀집(Dense) 벡터 전용 시맨틱 검색 테스트
* BM25 희소 키워드 가중치(텍스트 설명문 매칭)를 더하기 전에, `gemini-embedding-2`로 생성한 고차원 밀집 임베딩 벡터만을 활용하여 텍스트 쿼리와 비디오 청크 간의 순수 시맨틱 검색(유사도 매칭)이 얼마나 강력하게 작동하는지 먼저 단독으로 확인해 봅니다.
* 이 시맨틱 검색은 Gemini가 추출한 텍스트 묘사(설명문)와 상관없이, 비디오 장면과 텍스트의 순수 의미론적 관계성을 파악해 유사도를 계산합니다.


In [ ]:
# ===================================================
# 3. Dense-Only Semantic Search Demonstration (Interactive)
# ===================================================
import numpy as np
import base64
import os
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

def dense_search_only(query_text, top_k=3):
    """Performs semantic-only dense search using gemini-embedding-2."""
    if 'valid_chunks' not in globals() or not valid_chunks:
        print("인덱스에 등록된 비디오 청크가 존재하지 않습니다.")
        return []

    # Generate dense embedding for the text query
    query_dense = generate_multimodal_embedding(query_text, 'text')
    if query_dense is None:
        print("질의어 임베딩 변환 실패.")
        return []

    dense_scores = []
    for item in valid_chunks:
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            # Use cosine_similarity function defined in cell 15
            sim = cosine_similarity(query_dense, item['dense_embedding'])
            dense_scores.append(sim)
        else:
            dense_scores.append(0.0)

    # Rank results
    ranked_indices = np.argsort(dense_scores)[::-1][:top_k]

    results = []
    for idx in ranked_indices:
        results.append({
            "item": valid_chunks[idx],
            "score": dense_scores[idx]
        })
    return results

# Interactive Widget Setup
dense_query_input = widgets.Text(
    value='athlete tracking technology',
    placeholder='검색할 키워드를 입력하세요 (예: athlete, sports science, telemetry)',
    description='밀집 검색어:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

dense_search_button = widgets.Button(
    description='검색 실행',
    button_style='success'
)

dense_output_area = widgets.Output()

def on_dense_search_clicked(b):
    with dense_output_area:
        clear_output(wait=True)
        query = dense_query_input.value
        print(f"🔍 [순수 밀집(Dense) 벡터 검색 수행] | 질의어: '{query}'")
        
        results = dense_search_only(query, top_k=2)
        
        for i, res in enumerate(results):
            item = res['item']
            print(f"\n[결과 {i+1}위] 코사인 유사도 매칭 점수: {res['score']:.4f}")
            print(f"   청크 ID: {item['chunk_id']} (구간: {item['start_time']:.1f}초 - {item['end_time']:.1f}초)")
            desc_val = item.get('description', '아직 상세설명 텍스트(Gemini Flash 캡션)가 생성되지 않은 시점입니다.')
            print(f"   텍스트 설명(참고용): {desc_val}")
            
        if 'display_test_previews' in globals():
            display_test_previews(results, title=f"'{query}' 밀집 검색 결과")

dense_search_button.on_click(on_dense_search_clicked)

# Render widgets
display(widgets.HBox([dense_query_input, dense_search_button]), dense_output_area)

# Execute default search immediately on load
if 'valid_chunks' in globals() and valid_chunks:
    on_dense_search_clicked(None)
else:
    print("사용 가능한 청크 리스트가 없어 밀집 검색 테스트를 건너뜁니다.")

## 2단계: 하이브리드 검색 아키텍처 구현 (Hybrid Search)

밀도 높은 시맨틱 매칭과 키워드 기반 탐색을 결합한 엔터프라이즈급 하이브리드 검색기 파이프라인의 기초 모델을 시뮬레이션 환경으로 조립해 봅니다.


In [ ]:
def generate_chunk_description(video_path):
    """Generates a short description for a video chunk using Gemini 3.5 Flash."""
    try:
        with open(video_path, 'rb') as f:
            video_bytes = f.read()
        part = types.Part.from_bytes(data=video_bytes, mime_type='video/mp4')
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=[part, "Provide a short, detailed description of what is happening in this 10-second video clip. Focus on keywords, actions, and objects."],
            config=types.GenerateContentConfig(
                http_options=types.HttpOptions(
                    retry_options=types.HttpRetryOptions(
                        attempts=10,
                        initial_delay=1.0,
                        max_delay=3.0
                    )
                )
            )
        )
        return response.text
    except Exception as e:
        print(f"비디오 설명 묘사 텍스트 생성 실패 ({video_path}): {e}")
        return ""

# Generate descriptions for chunks
if 'valid_chunks' in globals() and valid_chunks:
    print(f"Gemini 3.5 Flash 모델을 사용하여 상위 {len(valid_chunks)}개 청크의 비디오 상세 설명 묘사 생성을 시작합니다...")
    for item in valid_chunks:
        local_path = item['file_path']
        print(f"처리 중: {local_path}...")
        description = generate_chunk_description(local_path)
        item['description'] = description
        print(f"생성된 텍스트: {description.strip()}")
else:
    print("유효 청크 데이터가 존재하지 않아 상세 묘사 설명 생성을 생략합니다.")

In [ ]:
import math
from collections import Counter

class SimpleBM25:
    """A simple implementation of BM25 for demonstration purposes."""
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.avgdl = sum(len(doc) for doc in corpus) / self.corpus_size if self.corpus_size > 0 else 0
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []

        for doc in corpus:
            self.doc_len.append(len(doc))
            self.doc_freqs.append(Counter(doc))

        for doc in corpus:
            for word in set(doc):
                self.idf[word] = self.idf.get(word, 0) + 1

        for word, freq in self.idf.items():
            self.idf[word] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1)

    def get_scores(self, query):
        scores = []
        for i in range(self.corpus_size):
            score = 0
            doc_freq = self.doc_freqs[i]
            doc_len = self.doc_len[i]
            for word in query:
                if word in doc_freq:
                    idf = self.idf.get(word, 0)
                    freq = doc_freq[word]
                    numerator = freq * (self.k1 + 1)
                    denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                    score += idf * numerator / denominator
            scores.append(score)
        return scores

def tokenize(text):
    return text.lower().split()

# Prepare corpus for BM25
corpus = []
tokenized_corpus = []

if 'valid_chunks' in globals() and valid_chunks:
    for item in valid_chunks:
        if 'description' in item and item['description']:
            corpus.append(item['description'])
            tokenized_corpus.append(tokenize(item['description']))

    if tokenized_corpus:
        bm25 = SimpleBM25(tokenized_corpus)
        print(f"{len(corpus)}개의 문서(Document) 기반 희소 검색용 BM25 인덱스가 성공적으로 빌드되었습니다.")
    else:
        print("BM25 인덱스 빌드에 필요한 설명 묘사 텍스트가 없습니다. 설명 생성 셀을 먼저 실행해 주세요.")
else:
    print("유효 청크 데이터가 없습니다.")


In [ ]:
import numpy as np
import base64
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

def hybrid_search(query_text, alpha=0.7, top_k=5):
    """Performs hybrid search combining dense and sparse results.

    Args:
        query_text: The search query string.
        alpha: Weight for dense results (0.7 favors dense, 0.6 favors sparse).
        top_k: Number of results to return.
    """
    if 'valid_chunks' not in globals() or not valid_chunks:
        print("인덱스에 등록된 비디오 청크가 존재하지 않습니다.")
        return []

    # 1. Dense Search
    query_dense = generate_multimodal_embedding(query_text, 'text')
    if query_dense is None:
        print("질의어 임베딩 변환 실패.")
        return []

    dense_scores = []
    for item in valid_chunks:
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            sim = cosine_similarity(query_dense, item['dense_embedding'])
            dense_scores.append(sim)
        else:
            dense_scores.append(0.0)

    # Normalize dense scores to [0, 1]
    dense_scores = np.array(dense_scores)
    if dense_scores.max() != dense_scores.min():
        dense_scores = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min())

    # 2. Sparse Search (BM25)
    tokenized_query = tokenize(query_text)
    sparse_scores = bm25.get_scores(tokenized_query)

    # Normalize sparse scores to [0, 1]
    sparse_scores = np.array(sparse_scores)
    if sparse_scores.max() != sparse_scores.min():
        sparse_scores = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min())

    # 3. Combine Scores
    combined_scores = alpha * dense_scores + (1 - alpha) * sparse_scores

    # Rank results
    ranked_indices = np.argsort(combined_scores)[::-1][:top_k]

    results = []
    for idx in ranked_indices:
        results.append({
            "chunk": valid_chunks[idx],
            "score": combined_scores[idx],
            "dense_score": dense_scores[idx],
            "sparse_score": sparse_scores[idx]
        })

    return results

def on_search_trigger(query_text, alpha=0.7):
    if not query_text.strip():
        print("검색어를 입력해 주세요.")
        return
        
    print(f"하이브리드 실시간 검색 수행 중: '{query_text}' (가중치 Alpha: {alpha} 고정)")
    results = hybrid_search(query_text, alpha=alpha, top_k=3)
    
    if not results:
        print("검색 결과가 존재하지 않습니다.")
        return
        
    for i, res in enumerate(results):
        chunk = res['chunk']
        print(f"\n검색 결과 {i+1}: 최종 가중 합산 점수: {res['score']:.3f} (밀집 시맨틱 점수: {res['dense_score']:.3f}, 희소 키워드 점수: {res['sparse_score']:.3f})")
        print(f"청크 ID: {chunk['chunk_id']}, 구간 타임스탬프: {chunk['start_time']:.1f}초 - {chunk['end_time']:.1f}초")
        print(f"청크 상세 설명: {chunk.get('description', '아직 상세 설명이 생성되지 않은 시점입니다.')}")

        # Play video chunk
        local_path = chunk['file_path']
        if os.path.exists(local_path):
            try:
                video_html = f'''
                <div style="position: relative; z-index: 9999; pointer-events: auto !important; display: inline-block;">
                    <video width="300" controls style="position: relative; z-index: 9999; pointer-events: auto !important; display: block;">
                        <source src="{local_path}" type="video/mp4">
                        Your browser does not support the video tag.
                    </video>
                </div>
                '''
                display(HTML(video_html))
            except Exception as e:
                print(f"대시보드 비디오 미디어 플레이어 기동 실패: {e}")
        else:
            print(f"지정된 로컬 경로에서 파일을 찾을 수 없습니다: {local_path}")

# Build minimalist interactive UI for Cell 22 (No Alpha Slider, default query: 'sports data science')
if 'valid_chunks' in globals() and valid_chunks:
    query_input = widgets.Text(
        value='sports data science',
        placeholder='검색어를 입력하고 엔터를 누르세요...',
        description='검색어:',
        continuous_update=False
    )
    
    out = widgets.Output()
    
    def run_interactive_search(change=None):
        with out:
            clear_output(wait=True)
            on_search_trigger(query_input.value, alpha=0.7)
            
    query_input.observe(run_interactive_search, names='value')
    
    display(widgets.VBox([query_input, out]))
    # Run initial search
    run_interactive_search()
else:
    print("사용 가능한 청크 리스트가 없어 검색 테스트를 건너뜁니다.")

# [Part 2] Managed Vector Search 2.0 & Scaling 🚀

이 파트에서는 로컬 시뮬레이션 환경을 넘어, 대규모 다종 미디어 코퍼스(Flickr 8k 이미지 및 다중 비디오 청크)를 구글 클라우드의 서버리스 고성능 인덱싱 엔진인 **Vector Search 2.0 (Serverless Collections)**에 실시간 동기화하고 상용화 수준으로 스케일링하는 과정을 학습합니다.


### 다종 미디어 통합 인덱싱 (Flickr 8k 이미지 및 비디오 코퍼스 결합)

보다 실전적이고 복합적인 시나리오 구현을 위해 이미지와 비디오 데이터셋을 결합한 통합 멀티모달 하이브리드 인덱싱을 준비합니다:
1.  **Flickr 8k 이미지 코퍼스**: 8,000여 개의 고해상도 일상 생활 사진 파일들과 설명문(Captions) 데이터셋.
2.  **비디오 청크 코퍼스**: 앞선 1단계에서 분할하고 임베딩 및 설명문을 임시 추출한 비디오 청크 세그먼트.

두 자산을 통합하여 동일한 시맨틱 공간 상에 나란히 배치함으로써, 이미지와 비디오를 가리지 않고 동시에 아우르는 하이브리드 크로스 미디어 검색엔진을 도출합니다.


In [ ]:
import pandas as pd
from google.cloud import storage
import io

# Read captions.txt from GCS
def read_gcs_file(bucket_name, blob_name):
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    content = blob.download_as_text()
    return content

try:
    bucket_name = "ai-multimodal-data"
    captions_csv = read_gcs_file(bucket_name, "Flickr 8k/captions.txt")
    df = pd.read_csv(io.StringIO(captions_csv))

    # Clean filenames
    df['image'] = df['image'].str.strip()
    df['caption'] = df['caption'].str.strip()

    # Group captions by image to have one document per image
    grouped = df.groupby('image')['caption'].apply(lambda x: " . ".join(x)).reset_index()
    image_docs = dict(zip(grouped['image'], grouped['caption']))
    
    print(f"Flickr 8k 캡션 메타데이터 로드 완료! 총 {len(image_docs)}개의 고해상도 이미지 매핑 테이블이 확보되었습니다.")
    print("데이터 샘플 (1003163366_44323f5815.jpg):\n", image_docs.get('1003163366_44323f5815.jpg'))
except Exception as e:
    print(f"Flickr 8k 파일 로드 오류: {e}")


### 🏷️ 사용자 정의 메타데이터 태깅 시뮬레이션 (Custom Tagging)

* 엔터프라이즈 하이브리드 검색의 가장 강력한 특징 중 하나는, AI가 자동으로 추출한 고차원 시맨틱 정보 외에 **사용자가 직접 입력한 커스텀 메타 태그(예: "Me", "Favorite", "Workspace")를 검색 인덱스에 결합**할 수 있다는 점입니다.
* 본 시뮬레이션 단계에서는 Flickr 이미지 보관함의 특정 사진 3개 상세설명에 문자열 태그 `"Me"`를 강제로 결합하고 BM25 희소 키워드 검색 인덱스를 재빌드합니다.
* 이를 통해 하이브리드 가중치(`alpha`) 값에 따라 일반 검색 결과와 개인화 태그 검색 결과가 어떻게 동적으로 상호작용하는지 실증합니다.


In [ ]:
# List of videos to process
videos = [
    "mlb_data_analysis.mp4",
    "team_usa_tech.mp4",
    "weather_climate_ml.mp4",
    "wildlife_recovery_ml.mp4"
]

In [ ]:
import pickle
import os
import urllib.request
from google.cloud import storage

combined_corpus = []
combined_tokenized = []
combined_registry = []
bucket_name = "ai-multimodal-data"
registry_file = 'full_dataset_registry.pkl'

# Download using public HTTP URL directly
PRECOMPUTED_GCS_URI = "https://storage.googleapis.com/ai-multimodal-data/full_dataset_registry.pkl"

def download_precomputed_data(uri, local_path):
    """Downloads pre-computed data from GCS or public HTTP URL."""
    try:
        if "TBD" in uri:
            print("다운로드 건너뜀: PRECOMPUTED_GCS_URI 변수를 유효한 경로로 업데이트해 주세요.")
            return False

        if uri.startswith("http://") or uri.startswith("https://"):
            print(f"퍼블릭 웹 URL을 통해 사전에 연산 완료된 가이드 아카이브 데이터를 다운로드합니다: {uri}...")
            urllib.request.urlretrieve(uri, local_path)
            print(f"로컬 {local_path} 경로로 다운로드가 정상 완료되었습니다.")
            return True

        if uri.startswith("gs://"):
            print(f"Cloud Storage 버킷에서 사전 연산 가이드 아카이브 복원을 시도합니다: {uri}...")
            parts = uri.replace("gs://", "").split("/", 1)
            b_name = parts[0]
            blob_name = parts[1]

            storage_client = storage.Client()
            bucket = storage_client.bucket(b_name)
            blob = bucket.blob(blob_name)
            blob.download_to_filename(local_path)
            print(f"로컬 {local_path} 경로로 다운로드가 정상 완료되었습니다.")
            return True
            
        print(f"지원하지 않는 프로토콜 주소 스키마입니다: {uri}")
        return False
    except Exception as e:
        print(f"다운로드 실패: {e}")
        return False

if not os.path.exists(registry_file):
    download_precomputed_data(PRECOMPUTED_GCS_URI, registry_file)

if os.path.exists(registry_file):
    print(f"아카이브 피클 파일({registry_file})로부터 레지스트리 목록을 로드하는 중...")
    try:
        with open(registry_file, 'rb') as f:
            combined_registry = pickle.load(f)
        print(f"총 {len(combined_registry)}개의 분석 자산(이미지 및 영상 청크) 목록이 메모리에 복원되었습니다.")

        for item in combined_registry:
            if 'description' in item and item['description']:
                combined_corpus.append(item['description'])
                combined_tokenized.append(tokenize(item['description']))
    except Exception as e:
        print(f"레지스트리 파일 복원 실패: {e}")
        os.remove(registry_file)

if not combined_registry:
    print(f"\n사전 구성된 아카이브가 없습니다. 워크숍 빠른 실습을 위해 서브셋 전처리(Subset mode)를 실행합니다...")
    IMAGE_LIMIT = 50
    count = 0

    if 'image_docs' in globals():
        for img_name, doc in image_docs.items():
            if count >= IMAGE_LIMIT:
                break
            gcs_path = f"gs://{bucket_name}/Flickr 8k/Images/{img_name}"
            embedding = generate_multimodal_embedding(gcs_path, 'image')
            if embedding is not None:
                combined_corpus.append(doc)
                combined_tokenized.append(tokenize(doc))
                combined_registry.append({
                    'id': img_name,
                    'type': 'image',
                    'path': gcs_path,
                    'description': doc,
                    'dense_embedding': embedding
                })
                count += 1
        print(f"레지스트리에 {count}개의 이미지가 성공적으로 편입되었습니다.")

    if 'all_video_chunks' in globals():
        for chunk in all_video_chunks:
            if 'description' in chunk and chunk['description'] and chunk['dense_embedding'] is not None:
                combined_corpus.append(chunk['description'])
                combined_tokenized.append(tokenize(chunk['description']))
                combined_registry.append({
                    'id': f"{chunk['source_video']}_{chunk['chunk_id']}",
                    'type': 'video_chunk',
                    'path': chunk['file_path'],
                    'description': chunk['description'],
                    'dense_embedding': chunk['dense_embedding']
                })
        print(f"레지스트리에 {len(all_video_chunks)}개의 비디오 청크 세그먼트가 편입되었습니다.")

if combined_tokenized:
    combined_bm25 = SimpleBM25(combined_tokenized)
    print(f"\n총 {len(combined_corpus)}개의 문서에 대한 대용량 통합 BM25 검색 인덱스가 빌드 완료되었습니다.")


In [ ]:
# Simulation: Adding "Me" tags to SPECIFIC user annotations
target_ids = [
    "1003163366_44323f5815.jpg",
    "1007129816_e794419615.jpg",
    "1015118661_980735411b.jpg"
]

TAG = "Me"
tagged_count = 0

for item in combined_registry:
    if item['id'] in target_ids:
        item['description'] += f" {TAG}"
        tagged_count += 1
        print(f"시뮬레이션 태그 부착 완료: {item['id']} -> 새 상세설명: {item['description']}")

print(f"\n특수 시뮬레이션 태그 '{TAG}'가 부착된 총 아이템 수: {tagged_count}")

# Display previews of the tagged images to make the setup visual for students
import ipywidgets as widgets
from IPython.display import display, Image as IPImage

previews = []
for item in combined_registry:
    if item['id'] in target_ids:
        path = item['path']
        try:
            if path.startswith("gs://"):
                from google.cloud import storage
                storage_client = storage.Client()
                parts = path[5:].split('/', 1)
                bucket = storage_client.bucket(parts[0])
                blob = bucket.blob(parts[1])
                image_bytes = blob.download_as_bytes()
                previews.append((item['id'], IPImage(data=image_bytes, width=180)))
            else:
                previews.append((item['id'], IPImage(filename=path, width=180)))
        except Exception as e:
            print(f"태그 이미지 프리뷰 로드 실패: {e}")

if previews:
    print(f"\n📷 [태그 '{TAG}'가 부착된 Flickr 이미지 사전 확인]:")
    outs = []
    for img_id, p in previews:
        o = widgets.Output()
        with o:
            display(p)
            print(f"ID: {img_id}")
        outs.append(o)
    display(widgets.HBox(outs))

# Rebuild BM25 index with the new tags
combined_corpus = []
combined_tokenized = []

for item in combined_registry:
    if 'description' in item and item['description']:
        combined_corpus.append(item['description'])
        combined_tokenized.append(tokenize(item['description']))

if combined_tokenized:
    combined_bm25 = SimpleBM25(combined_tokenized)
    print(f"\n태그 정보를 포괄한 {len(combined_corpus)}개 통합 BM25 인덱스 재빌드를 완료했습니다.")


In [ ]:
def combined_hybrid_search(query_text, alpha=0.7, top_k=5):
    """Performs hybrid search across combined images and video chunks."""
    if 'combined_registry' not in globals() or not combined_registry:
        print("색인 등록된 데이터 항목이 존재하지 않습니다.")
        return []

    # 1. Dense Search
    query_dense = generate_multimodal_embedding(query_text, 'text')
    if query_dense is None:
        print("질의어 임베딩 변환 실패.")
        return []

    dense_scores = []
    for item in combined_registry:
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            sim = cosine_similarity(query_dense, item['dense_embedding'])
            dense_scores.append(sim)
        else:
            dense_scores.append(0.0)

    dense_scores = np.array(dense_scores)
    if dense_scores.max() != dense_scores.min():
        dense_scores = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min())

    # 2. Sparse Search (BM25)
    tokenized_query = tokenize(query_text)
    sparse_scores = combined_bm25.get_scores(tokenized_query)

    sparse_scores = np.array(sparse_scores)
    if sparse_scores.max() != sparse_scores.min():
        sparse_scores = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min())

    # 3. Combine Scores
    combined_scores = alpha * dense_scores + (1 - alpha) * sparse_scores

    # Rank results
    ranked_indices = np.argsort(combined_scores)[::-1][:top_k]

    results = []
    for idx in ranked_indices:
        results.append({
            "item": combined_registry[idx],
            "score": combined_scores[idx],
            "dense_score": dense_scores[idx],
            "sparse_score": sparse_scores[idx]
        })

    return results

def search_videos_only(query_text, alpha=0.7, top_k=5):
    """Performs search and filters results to keep only video chunks."""
    if 'combined_registry' not in globals() or not combined_registry:
        return []
    # Retrieve a larger pool of candidates from the full index to ensure we capture enough video chunks
    raw_results = combined_hybrid_search(query_text, alpha=alpha, top_k=200)
    # Filter candidates to keep only video chunks post-search
    video_results = [r for r in raw_results if r['item']['type'] == 'video_chunk']
    return video_results[:top_k]

if 'combined_registry' in globals() and combined_registry:
    # Test 1: Standard query
    query = "child in pink dress"
    print(f"통합 하이브리드 검색 수행 중: '{query}'")
    results = combined_hybrid_search(query, alpha=0.7)

    for i, res in enumerate(results):
        item = res['item']
        print(f"\n검색 결과 {i+1}: 최종 가중 합산 점수: {res['score']:.3f} (밀집 시맨틱 점수: {res['dense_score']:.3f}, 희소 키워드 점수: {res['sparse_score']:.3f})")
        print(f"미디어 형태: {item['type']}, 식별 아이디: {item['id']}")
        short_desc = item['description'][:100] + "..." if len(item['description']) > 100 else item['description']
        print(f"상세 설명: {short_desc}")

    if 'display_test_previews' in globals():
         display_test_previews(results, title="pink dress 검색 매칭 결과")

    # Test 2: Tag weight comparison
    query = "Me at the beach"
    print(f"\n==================================================")
    print(f"실험 A: Alpha = 0.8 (밀집 시맨틱 유사도 중심 검색) | 질의어: '{query}'")
    results_alpha_high = combined_hybrid_search(query, alpha=0.8, top_k=5)
    for i, res in enumerate(results_alpha_high):
        item = res['item']
        short_desc = item['description'][:100] + "..." if len(item['description']) > 100 else item['description']
        print(f"  [{i+1}위] 점수: {res['score']:.3f} | {item['id']} ({item['type']})")
        print(f"         설명: {short_desc}")
    if 'display_test_previews' in globals():
        display_test_previews(results_alpha_high, title="Alpha = 0.8 (밀집 유사도 중심) 결과")

    print(f"\n--------------------------------------------------")
    print(f"실험 B: Alpha = 0.3 (희소 키워드 태그 중심 검색) | 질의어: '{query}'")
    results_alpha_low = combined_hybrid_search(query, alpha=0.3, top_k=5)
    for i, res in enumerate(results_alpha_low):
        item = res['item']
        short_desc = item['description'][:100] + "..." if len(item['description']) > 100 else item['description']
        print(f"  [{i+1}위] 점수: {res['score']:.3f} | {item['id']} ({item['type']})")
        print(f"         설명: {short_desc}")
    if 'display_test_previews' in globals():
        display_test_previews(results_alpha_low, title="Alpha = 0.3 (희소 태그 중심) 결과")


## 3단계: 검색 결과 최적화 (Post-Search Optimization)

하이브리드 검색 엔진에서 일차적으로 걸러져 반환된 상위 후보군(Candidates)들의 시각적 다양성 및 검색 품질을 비즈니스 요구사항에 맞춰 보정하는 단계입니다.

### 리랭킹 아키텍처 (Reranking with VertexRanker)
* 초기 벡터 검색(코사인 유사도 매칭)만으로는 확보하기 힘든 **미세 도메인 맥락 정렬**을 고도화하기 위해, 양방향 주의집중 기법을 사용하는 **Agent Search Ranking API**를 연동합니다.
* 1차 후보군 데이터 목록을 관리형 Reranker 서비스에 전송하여 실시간 고정밀 재점수(Re-scoring)를 계산하고 최종 정렬 순위를 갱신합니다.

### 비디오 크라우딩 방지 (Video Crowding Mitigation)
* 동일한 비디오 출처의 중복되거나 유사한 시점의 청크들이 상위 목록을 전부 독점하는 현상을 방지하기 위해 **개별 비디오 소스당 노출 개수 상한선을 설정(Crowding Filter)**하는 구조를 설계합니다.
* 이를 통해 한 영상에서만 씬이 줄지어 나오는 쏠림 현상을 예방하고, 검색 유저에게 보다 다채로운 비디오 소스 후보들을 한눈에 제시할 수 있습니다.


In [ ]:
# VertexRanker (Pipeline Step 1: Reranking)
from google.cloud import discoveryengine_v1 as discoveryengine
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import base64
import os
import threading

def rerank_results(query, results):
    """Calls Agent Search Ranking API for re-scoring results."""
    print(f"Agent Search Ranking API를 호출하여 검색 질의('{query}') 고성능 리랭킹을 요청합니다...")
    print(f"재정렬(Reranking)을 위해 상위 {len(results)}개의 검색 후보군 데이터를 전송합니다...")

    try: 
        rank_client = discoveryengine.RankServiceClient()
        request = discoveryengine.RankRequest(
            ranking_config=f"projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config",
            model="default",
            query=query,
            records=[
                discoveryengine.RankingRecord(
                    id=r['item']['id'].replace(".", "_").replace("-", "_").replace(" ", "_"),
                    title=r['item'].get('type', ''),
                    content=r['item'].get('description', '')
                ) for r in results
            ]
        )

        response = rank_client.rank(request)
        print("Ranking API 호출 및 재점수 연산이 완료되었습니다.")

        reranked_results = []
        for record in response.records:
            orig = next((r for r in results if r['item']['id'].replace(".", "_").replace("-", "_").replace(" ", "_") == record.id), None)
            if orig:
                new_res = orig.copy()
                new_res['rerank_score'] = record.score
                new_res['score'] = record.score
                reranked_results.append(new_res)
        return reranked_results
    except Exception as e:
        print(f"Agent Search Ranking API 호출 실패: {e}")
        return results

# Interactive Reranker Widget Setup
rerank_query_input = widgets.Text(
    value='weather prediction model',
    placeholder='리랭킹을 수행할 검색어를 입력하세요 (예: weather, wildlife recovery, athlete tech)...',
    description='리랭크 검색어:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

rerank_search_button = widgets.Button(
    description='리랭킹 실행',
    button_style='success'
)

rerank_output_area = widgets.Output()

# Busy flag to prevent concurrent execution race conditions
is_rerank_searching = False

def on_rerank_search_clicked(b):
    global reranked_results, is_rerank_searching
    if is_rerank_searching:
        return
    is_rerank_searching = True
    
    try:
        with rerank_output_area:
            clear_output(wait=True)
            query = rerank_query_input.value
            print(f"비디오 청크만을 대상으로 검색을 수행합니다 (재정렬 검증) | 질의어: '{query}'")
            
            results_for_crowding = search_videos_only(query, alpha=0.7, top_k=10)
            
            if results_for_crowding:
                # Show BEFORE reranking previews (Raw initial search rankings)
                if 'display_test_previews' in globals():
                    display_test_previews(results_for_crowding, title="Reranking 이전 (1차 하이브리드 검색 순위)")
                    
                print("\n" + "="*50)
                reranked_results = rerank_results(query, results_for_crowding)
                
                print("\n[Reranking 고정밀 재정렬 결과 (순위 변동 확인)]:")
                for i, res in enumerate(reranked_results):
                    item = res['item']
                    short_desc = item['description'][:100] + "..." if len(item['description']) > 100 else item['description']
                    print(f"  랭킹 {i+1}위: {item['id']} (리랭킹 환산 점수: {res['rerank_score']:.4f}) | 설명: {short_desc}")
                    
                # Show AFTER reranking previews (Re-scored and re-sorted rankings)
                if 'display_test_previews' in globals():
                    display_test_previews(reranked_results, title="Reranking 이후 (재정렬 정밀 순위)")
            else:
                print("검색 결과 후보군이 존재하지 않아 리랭킹을 건너뜀.")
    finally:
        is_rerank_searching = False

rerank_search_button.on_click(on_rerank_search_clicked)

# Render widgets
if 'combined_registry' in globals() and combined_registry:
    display(widgets.HBox([rerank_query_input, rerank_search_button]), rerank_output_area)
    # Run default query on load asynchronously
    threading.Timer(0.1, lambda: on_rerank_search_clicked(None)).start()
else:
    print("통합 레지스트리가 로드되지 않아 리랭크 테스트를 스킵합니다.")

In [ ]:
# Video Crowding Filter (Pipeline Step 2: Crowding Mitigation)
def apply_video_crowding(search_results, max_per_video=2):
    """Filters search results to limit the number of chunks from the same video."""
    filtered_results = []
    video_counts = {} 

    for res in search_results:
        item = res['item']
        if item['type'] == 'video_chunk':
            video_id = item['id'].split('_')[0]
            count = video_counts.get(video_id, 0)
            if count < max_per_video:
                filtered_results.append(res)
                video_counts[video_id] = count + 1
            else:
                print(f"크라우딩 필터: 비디오 소스 {video_id}의 청크 {item['id']} 노출 제외 처리 (개수 제한 도달)")
        else:
            filtered_results.append(res)
    return filtered_results

if 'reranked_results' in globals() and reranked_results:
    # Limit same-video chunks to at most 2 to showcase crowding filter
    print("동일 비디오 과다 노출 제어 알고리즘(Video Crowding Filter, 영상별 최대 2개 노출)을 적용합니다...")
    crowded_results = apply_video_crowding(reranked_results, max_per_video=2)
    
    print(f"\n크라우딩 제어 반영 후 최종 표시 결과 수: {len(crowded_results)}")
    for i, res in enumerate(crowded_results):
        item = res['item']
        # Shorten description output for console readability
        short_desc = item['description'][:100] + "..." if len(item['description']) > 100 else item['description']
        print(f"표시 결과 {i+1}: {item['id']} (유사도/리랭크 합산 점수: {res['score']:.3f})")
        print(f"   상세 설명: {short_desc}")
        
    if 'display_test_previews' in globals():
        display_test_previews(crowded_results, title="최종 크라우딩 필터 차단 후 비디오 프리뷰")


### 💡 워크숍 진행을 위한 코드 최적화 및 가속 팁

본 워크숍 실습에서는 원활하고 경제적인 컴퓨팅 제어를 위해 아래와 같은 실무 튜닝이 반영되었습니다:
1.  **로컬 캐싱 정책**: 미디어 파일을 불필요하게 중복 다운로드하지 않도록 파일 시스템 존재 여부를 검사 후 선별 로드합니다.
2.  **데이터 서브셋 최적화**: 런타임 타임아웃과 과도한 API 트래픽 비용을 차단하기 위해 실습에 최적화된 소규모 서브셋만 색인에 활용합니다.
3.  **대체 리로드 전략 (Fallback Strategy)**: 실시간 연산 과정이 원격 환경 제약 등으로 오래 걸리는 경우, 미리 생성된 레지스트리 피클 아카이브(`full_dataset_registry.pkl`)를 파일 시스템이나 버킷에서 다이렉트로 복원하여 흐름을 중단 없이 이어가게 설계했습니다.

**프로덕션 스케일링 고려사항 (Production Scale-out):**
엔터프라이즈 상용 시스템으로의 마이그레이션 시에는 다음과 같은 분산/병렬 아키텍처 요소를 고려해야 합니다:
*   **병렬 배치 처리 (Parallel processing)**: `concurrent.futures` 등을 적용하여 대량의 제미나이 임베딩/텍스트 생성 요청을 쓰레드 풀 단위로 다중 병렬 처리하여 대기시간을 대폭 감소시킵니다.
*   **일괄 배치 API (Batch API)**: 가능한 경우 대규모 비동기 데이터 일괄 분석 API 파이프라인을 호출하여 일률 색인을 실행합니다.
*   **분산 처리 파이프라인**: 테라바이트급 대규모 저장소 이관 시에는 Google Cloud Dataflow(Apache Beam 기반) 또는 Ray 클러스터를 통해 전체 전처리를 수백 개의 병렬 노드로 스케일 아웃시킵니다.


## 4단계: 벡터 검색 상용화 단계 (Vector Search 2.0 구현)

이 섹션에서는 대용량 고차원 벡터에 대해 10ms 단위의 초고속 유사도 매칭을 보장하는 서버리스 매니지드 데이터베이스 솔루션인 **Vector Search 2.0 (Serverless Collections)**을 연동합니다.


In [ ]:
# Provisioning Vector Search 2.0 Collection
from google.cloud import vectorsearch_v1beta as vectorsearch
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# PROJECT_ID is inherited from Cell 4
REGION = "us-central1"
COLLECTION_ID = "multimodal-media-collection"

print("관리형 Vector Search 2.0 서버리스 클라이언트 초기화 중...")
try:
    vector_search_client = vectorsearch.VectorSearchServiceClient()
    parent = f"projects/{PROJECT_ID}/locations/{REGION}"
    collection_name = f"{parent}/collections/{COLLECTION_ID}"

    data_schema = {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "filename": {"type": "string"},
            "media_type": {"type": "number"},
            "start": {"type": "number"},
            "end": {"type": "number"},
            "description": {"type": "string"}
        }
    }

    vector_schema = {
        "content_embedding": vectorsearch.VectorField(
            dense_vector=vectorsearch.DenseVectorField(
                dimensions=3072,
                vertex_embedding_config=vectorsearch.VertexEmbeddingConfig(
                    model_id="gemini-embedding-2"
                )
            )
        ),
        "description_embedding": vectorsearch.VectorField(
            sparse_vector=vectorsearch.SparseVectorField()
        )
    }

    collection_config = vectorsearch.Collection(
        data_schema=data_schema,
        vector_schema=vector_schema
    )

    create_request = vectorsearch.CreateCollectionRequest(
        parent=parent,
        collection_id=COLLECTION_ID,
        collection=collection_config
    )

    # 1. Check if collection already exists
    collection_exists = False
    existing_coll = None
    try:
        existing_coll = vector_search_client.get_collection(name=collection_name)
        collection_exists = True
        print(f"ℹ️ 알림: 동일한 ID의 컬렉션이 구글 클라우드에 이미 존재합니다: {COLLECTION_ID}")
    except Exception:
        pass

    # 2. Interactive Choice UI if collection exists
    if collection_exists:
        print("⚠️ 아래 버튼 중 하나를 클릭하여 진행 방향을 선택해 주세요:")
        
        button_reuse = widgets.Button(description="기존 컬렉션 그대로 사용 (권장)", button_style='success', layout=widgets.Layout(width='250px'))
        button_delete = widgets.Button(description="삭제 후 새 컬렉션으로 재생성", button_style='danger', layout=widgets.Layout(width='250px'))
        button_local = widgets.Button(description="로컬 인메모리 시뮬레이션 진행", button_style='warning', layout=widgets.Layout(width='250px'))
        
        output_choice = widgets.Output(layout=widgets.Layout(margin='10px 0'))
        
        def on_reuse_clicked(b):
            global collection
            with output_choice:
                clear_output()
                collection = existing_coll
                print(f"✅ 기존 컬렉션을 활성화하여 연결했습니다: {collection.name}")
                print("이전 업로드된 데이터가 그대로 유지됩니다. 바로 다음 쿼리 실습 셀로 넘어가셔도 좋습니다.")
                
        def on_delete_clicked(b):
            global collection
            with output_choice:
                clear_output()
                print("기존 컬렉션을 구글 클라우드에서 삭제하는 중입니다... (약 1분 소요)")
                try:
                    try:
                        if 'combined_registry' in globals() and combined_registry:
                            print("1. 기존 컬렉션 내부의 데이터 오브젝트들을 소거하는 중...")
                            data_client = vectorsearch.DataObjectServiceClient()
                            dp_ids = [item['id'].replace(".", "_").replace("-", "_").replace(" ", "_") for item in combined_registry]
                            
                            batch_size = 500
                            for k in range(0, len(dp_ids), batch_size):
                                packet = dp_ids[k:k+batch_size]
                                
                                delete_reqs = [
                                    vectorsearch.DeleteDataObjectRequest(
                                        name=f"{collection_name}/dataObjects/{dp_id}"
                                    ) for dp_id in packet
                                ]
                                
                                req = vectorsearch.BatchDeleteDataObjectsRequest(
                                    parent=collection_name,
                                    requests=delete_reqs
                                )
                                data_client.batch_delete_data_objects(request=req)
                            print("   데이터 오브젝트 소거 완료.")
                    except Exception as clean_err:
                        print(f"   경고: 데이터 오브젝트 소거 단계 건너뜀 (계속 진행): {clean_err}")

                    print("2. 컬렉션 해체를 시작합니다...")
                    operation = vector_search_client.delete_collection(name=collection_name)
                    operation.result()
                    print("삭제 완료. 새 빈 컬렉션을 백그라운드 생성 중입니다...")
                    
                    operation = vector_search_client.create_collection(request=create_request)
                    collection = operation.result()
                    print(f"✅ 새 서버리스 컬렉션 프로비저닝 완료: {collection.name}")
                    print("인덱스가 비어 있습니다. 다음 셀에서 임베딩 업서트를 실행해 주세요.")
                except Exception as err:
                    print(f"삭제 후 재프로비저닝 실패: {err}")
                    print("안전을 위해 로컬 시뮬레이션 모드로 대체 우회합니다.")
                    collection = None
                    
        def on_local_clicked(b):
            global collection
            with output_choice:
                clear_output()
                collection = None
                print("⚠️ 로컬 시뮬레이션 모드로 전환했습니다 (원격 DB 연결 안 함).")
                print("클라우드 과금 없이 실습용 코드를 가상 로컬 모드로 돌립니다.")
                
        button_reuse.on_click(on_reuse_clicked)
        button_delete.on_click(on_delete_clicked)
        button_local.on_click(on_local_clicked)
        
        display(widgets.VBox([
            widgets.HBox([button_reuse, button_delete, button_local]),
            output_choice
        ]))
        
        # Default initialization: Reuse to prevent notebook breakage if cells run sequentially without interaction
        collection = existing_coll
    else:
        print("서버리스 컬렉션 프로비저닝을 시작합니다... (최초 생성 시 1~2분 소요)")
        operation = vector_search_client.create_collection(request=create_request)
        collection = operation.result()
        print(f"✅ Vector Search 2.0 서버리스 컬렉션 프로비저닝 완료: {collection.name}")

except Exception as e:
    print(f"Vector Search 2.0 매니지드 컬렉션 연결 및 생성 실패: {e}")
    print("로컬 인메모리 시뮬레이션 모드로 전환하여 실습을 계속 진행합니다.")
    collection = None


### 4단계 (b): 인덱스 생성 및 실시간 쿼리 매칭 (Indexing & Search)

구축된 서버리스 벡터 컬렉션에 로컬에서 정비된 임베딩 페이로드들을 실시간 업서트(Upsert)하고, 다차원 쿼리에 대한 최적화된 검색 인터페이스를 정의합니다.


In [ ]:
import numpy as np
from google.cloud import vectorsearch_v1beta as vectorsearch
from concurrent.futures import ThreadPoolExecutor

# Phase 4b: Indexing and Searching using Vector Search 2.0 Collections

print("고성능 병렬 스레드 배치를 기동하여 Vector Search 2.0 컬렉션으로 임베딩 업서트를 실행합니다...")
datapoints_count = 0
if 'collection' in globals() and collection and 'combined_registry' in globals() and combined_registry:
    try:
        data_client = vectorsearch.DataObjectServiceClient()
        
        # Prepare all requests
        requests = []
        for item in combined_registry:
            if 'dense_embedding' in item and item['dense_embedding'] is not None:
                dp_id = item['id'].replace(".", "_").replace("-", "_").replace(" ", "_")
                data_obj = vectorsearch.DataObject(
                    data_object_id=dp_id,
                    data={
                        "title": item.get('title', ''),
                        "filename": item.get('filename', ''),
                        "media_type": 1 if item['type'] == 'image' else 0,
                        "start": item.get('start_time', 0.0),
                        "end": item.get('end_time', 0.0),
                        "description": item.get('description', '')
                    },
                    vectors={
                        "content_embedding": vectorsearch.Vector(
                            dense=vectorsearch.DenseVector(values=item['dense_embedding'])
                        )
                    }
                )
                
                requests.append(
                    vectorsearch.CreateDataObjectRequest(
                        parent=collection.name,
                        data_object_id=dp_id,
                        data_object=data_obj
                    )
                )
        
        # Execute batch upsert concurrently (batch size 250 is the API limit)
        batch_size = 250
        batches = [requests[i:i+batch_size] for i in range(0, len(requests), batch_size)]
        
        def send_batch(batch_idx_and_reqs):
            batch_idx, batch_reqs = batch_idx_and_reqs
            batch_request = vectorsearch.BatchCreateDataObjectsRequest(
                parent=collection.name,
                requests=batch_reqs
            )
            try:
                data_client.batch_create_data_objects(request=batch_request)
                print(f"배치 패킷 {batch_idx + 1} 업서트 완료 ({len(batch_reqs)}개 오브젝트 전송)...")
            except Exception as e:
                # Handle cases where data objects already exist
                if "already exists" in str(e).lower():
                    print(f"배치 패킷 {batch_idx + 1}: 업로드하려는 항목이 컬렉션 내에 이미 존재합니다 (중복 우회 및 스킵).")
                else:
                    raise e
            return len(batch_reqs)
            
        with ThreadPoolExecutor(max_workers=min(4, len(batches) if len(batches) > 0 else 1)) as executor:
            results = list(executor.map(send_batch, enumerate(batches)))
            datapoints_count = sum(results)
            
        print(f"총 {datapoints_count}개의 멀티모달 데이터 오브젝트를 Vector Search 2.0 관리형 컬렉션에 업로드 반영했습니다.")
    except Exception as e:
        print(f"Vector Search 2.0 컬렉션 업서트 실패: {e}")
else:
    print("Vector Search 2.0 컬렉션이 정의되지 않아 데이터 전송을 생략합니다.")

def find_similar_content(query_input, query_type, top_k=5):
    """Searches for similar content using Vector Search 2.0."""
    if 'collection' not in globals() or not collection:
        print("Vector Search 2.0 서버리스 DB를 사용할 수 없어 로컬 가상 환경 모드로 대체 실행합니다.")
        # Local simulation mode
        if query_type == 'text':
            if 'combined_hybrid_search' in globals():
                raw_results = combined_hybrid_search(query_input, alpha=0.7, top_k=top_k)
                return [{
                    'id': r['item']['id'],
                    'score': r['score'],
                    'type': r['item']['type'],
                    'path': r['item']['path']
                } for r in raw_results]
            return []
        else:
            # Multi-modal query local simulation (Dense search only)
            query_embedding = generate_multimodal_embedding(query_input, query_type)
            if query_embedding is None:
                return []
            
            # Manual cosine similarity calculation across registry
            dense_scores = []
            for item in combined_registry:
                if 'dense_embedding' in item and item['dense_embedding'] is not None:
                    sim = cosine_similarity(query_embedding, item['dense_embedding'])
                    dense_scores.append(sim)
                else:
                    dense_scores.append(0.0)
            
            ranked_indices = np.argsort(dense_scores)[::-1][:top_k]
            results = []
            for idx in ranked_indices:
                results.append({
                    'id': combined_registry[idx]['id'],
                    'score': dense_scores[idx],
                    'type': combined_registry[idx]['type'],
                    'path': combined_registry[idx]['path']
                })
            return results

    try:
        search_client = vectorsearch.DataObjectSearchServiceClient()
        if query_type == 'text':
            # Task type must be specified for SemanticSearch (e.g., QUESTION_ANSWERING)
            request = vectorsearch.SearchDataObjectsRequest(
                parent=collection.name,
                semantic_search=vectorsearch.SemanticSearch(
                    search_text=query_input,
                    search_field="content_embedding",
                    task_type="QUESTION_ANSWERING",
                    top_k=top_k
                )
            )
        else:
            query_embedding = generate_multimodal_embedding(query_input, query_type)
            if query_embedding is None:
                return []
            # vector and top_k are defined inside VectorSearch.
            request = vectorsearch.SearchDataObjectsRequest(
                parent=collection.name,
                vector_search=vectorsearch.VectorSearch(
                    vector=vectorsearch.DenseVector(values=query_embedding),
                    search_field="content_embedding",
                    top_k=top_k
                )
            )

        response = search_client.search_data_objects(request=request)
        results = []
        for match in response.results:
            item = next((x for x in combined_registry if x['id'].replace(".", "_").replace("-", "_").replace(" ", "_") == match.data_object.data_object_id), None)
            if item:
                results.append({
                    'id': item['id'],
                    'score': match.distance,
                    'type': item['type'],
                    'path': item['path']
                })
        return results
    except Exception as e:
        print(f"Vector Search 2.0 API 조회 연산 실패: {e}")
        return []

if 'collection' in globals() and collection:
    print("\nVector Search 2.0 매니지드 컬렉션 실시간 질의 테스트: 'lemons'")
    try:
        test_results = find_similar_content('lemons', 'text', top_k=3)
        for res in test_results:
            print(f"유사 씬 매칭 성공: {res['id']} (유사도 거리 점수: {res['score']:.4f})")
    except Exception as e:
        print(f"컬렉션 질의 테스트 실패: {e}")


## 5단계: 멀티모달 하이브리드 검색 대시보드 UI (Interactive Search UI)

사용자가 텍스트, 이미지, 비디오, 오디오 등 이종 멀티모달 조건으로 검색을 직접 질의해볼 수 있는 인터랙티브 GUI 컴포넌트를 `ipywidgets` 기반으로 렌더링합니다.

또한, **알파 슬라이더(Alpha Parameter Slider)**를 통해 실시간 밀집 검색(Dense, 시맨틱)과 희소 검색(Sparse, 키워드) 가중치 변동이 검색 목록에 어떤 구조적 영향을 미치는지 실시간 데모로 시각화합니다.


In [ ]:
# ===================================================
# INTERACTIVE MULTIMODAL SEARCH DASHBOARD UI
# ===================================================
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage, HTML
import random
import os
import base64
import numpy as np

# 0. Helper definition for find_similar_content if Cell 38 was skipped
if 'find_similar_content' not in globals():
    def find_similar_content(query_input, query_type, top_k=5):
        """Local simulation fallback if managed Vector Search cell wasn't run."""
        if 'combined_registry' not in globals() or not combined_registry:
            print("레지스트리 데이터가 로드되지 않았습니다.")
            return []
            
        if query_type == 'text':
            if 'combined_hybrid_search' in globals():
                raw_results = combined_hybrid_search(query_input, alpha=0.7, top_k=top_k)
                return [{
                    'id': r['item']['id'],
                    'score': r['score'],
                    'type': r['item']['type'],
                    'path': r['item']['path']
                } for r in raw_results]
            return []
        else:
            query_embedding = generate_multimodal_embedding(query_input, query_type)
            if query_embedding is None:
                return []
            
            # Manual cosine similarity
            dense_scores = []
            for item in combined_registry:
                if 'dense_embedding' in item and item['dense_embedding'] is not None:
                    # Use cosine_similarity function defined in cell 15
                    sim = cosine_similarity(query_embedding, item['dense_embedding'])
                    dense_scores.append(sim)
                else:
                    dense_scores.append(0.0)
            
            ranked_indices = np.argsort(dense_scores)[::-1][:top_k]
            results = []
            for idx in ranked_indices:
                results.append({
                    'id': combined_registry[idx]['id'],
                    'score': dense_scores[idx],
                    'type': combined_registry[idx]['type'],
                    'path': combined_registry[idx]['path']
                })
            return results

# 1. UI Components Instantiation
query_type_dropdown = widgets.Dropdown(
    options=['text', 'image', 'video', 'audio'],
    value='text',
    description='Query Type:',
)

text_query_input = widgets.Text(
    value='wildlife animal tracking',
    placeholder='Enter text query',
    description='Text Query:',
    disabled=False
)

alpha_slider = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Alpha (Dense vs Sparse):',
    style={'description_width': 'initial'}
)

upload_button = widgets.FileUpload(
    accept='image/*,video/*,audio/*',
    multiple=False,
    description='Upload File'
)

shuffle_button = widgets.Button(
    description='Shuffle Gallery Query',
    button_style='info'
)

search_button = widgets.Button(
    description='Search',
    button_style='success'
)

output_area = widgets.Output()

# 2. display_media_results Helper Definition with Double Fallback Path Resolvers
def display_media_results(results):
    """Helper function to display results with improved layout and deduplication."""
    seen_ids = set()
    deduped_results = []
    for res in results:
        if res['id'] not in seen_ids:
            deduped_results.append(res)
            seen_ids.add(res['id'])

    images = [res for res in deduped_results if res['type'] == 'image']
    videos = [res for res in deduped_results if res['type'] == 'video_chunk']

    print(f"\n상위 검색 결과 매칭 (중복 정비 후 실질 매칭 {len(deduped_results)}개):")

    # --- Enforce counts ---
    display_images = images[:3]
    if len(display_images) < 3:
        found_ids = {res['id'] for res in deduped_results}
        if 'combined_registry' in globals():
            for item in combined_registry:
                if item['type'] == 'image' and item['id'] not in found_ids:
                    display_images.append({
                        'id': item['id'],
                        'score': 0.0,
                        'type': 'image',
                        'path': item['path']
                    })
                    found_ids.add(item['id'])
                    if len(display_images) >= 3:
                        break

    display_videos = videos[:1]
    if not display_videos:
        if 'combined_registry' in globals():
            for item in combined_registry:
                if item['type'] == 'video_chunk':
                    display_videos.append({
                        'id': item['id'],
                        'score': 0.0,
                        'type': item['type'],
                        'path': item['path']
                    })
                    break
    # ----------------------

    # Display Images
    img_widgets = []
    for img in display_images[:3]:
        out = widgets.Output()
        with out:
            if img['score'] > 0:
                print(f"밀집 점수: {img['score']:.4f} | {img['id']}")
            else:
                print(f"보관함 대체 매핑 | {img['id']}")

            path = img['path']
            try:
                image_bytes = None
                # Try original path
                if path.startswith("gs://"):
                    from google.cloud import storage
                    storage_client = storage.Client()
                    parts = path[5:].split('/', 1)
                    b_name = parts[0]
                    bl_name = parts[1] if len(parts) > 1 else ""
                    bucket = storage_client.bucket(b_name)
                    blob = bucket.blob(bl_name)
                    image_bytes = blob.download_as_bytes()
                else:
                    if os.path.exists(path):
                        with open(path, 'rb') as f:
                            image_bytes = f.read()
                            
                if image_bytes:
                    display(IPImage(data=image_bytes, width=250))
                else:
                    print("[이미지 파일 없음]")
            except Exception as e:
                print(f"대시보드 이미지 뷰어 렌더링 실패: {e}")
        img_widgets.append(out)

    if img_widgets:
        print("\n최종 매칭 이미지 프리뷰:")
        display(widgets.HBox(img_widgets))

    # Display Video
    if display_videos:
        v = display_videos[0]
        if v['score'] > 0:
            print(f"\n최종 매칭 비디오 세그먼트 프리뷰 - 매칭 점수: {v['score']:.4f} | {v['id']}")
        else:
            print(f"\n최종 비디오 프리뷰 - 보관함 대체 매핑 (검색 제외 상태) | {v['id']}")

        path = v['path']
        try:
            video_bytes = None
            
            # Step 1: Try path as written
            if path.startswith("gs://"):
                from google.cloud import storage
                storage_client = storage.Client()
                parts = path[5:].split('/', 1)
                b_name = parts[0]
                bl_name = parts[1] if len(parts) > 1 else ""
                bucket = storage_client.bucket(b_name)
                blob = bucket.blob(bl_name)
                video_bytes = blob.download_as_bytes()
            else:
                if os.path.exists(path):
                    with open(path, 'rb') as f:
                        video_bytes = f.read()

            # Step 2: Try local chunks folder
            if video_bytes is None:
                try:
                    if '_' in v['id']:
                        chunk_idx = v['id'].rsplit('_', 1)[1]
                    else:
                        chunk_idx = v['id']
                    
                    local_fallback_path = f"local_chunks/chunk_{chunk_idx}.mp4"
                    if os.path.exists(local_fallback_path):
                        with open(local_fallback_path, 'rb') as f:
                            video_bytes = f.read()
                        print(f"      (로컬 백업 매핑 적용: {local_fallback_path})")
                except Exception:
                    pass

            # Step 3: Try GCS bucket folder
            if video_bytes is None:
                try:
                    if '_' in v['id']:
                        parts = v['id'].rsplit('_', 1)
                        video_filename = parts[0]
                        chunk_idx = parts[1]
                    else:
                        video_filename = "team_usa_tech.mp4"
                        chunk_idx = v['id']
                        
                    from google.cloud import storage
                    storage_client = storage.Client()
                    bucket = storage_client.bucket("ai-multimodal-data")
                    blob_name = f"vid-chunks/{video_filename}_chunks/chunk_{chunk_idx}.mp4"
                    
                    blob = bucket.blob(blob_name)
                    if blob.exists():
                        video_bytes = blob.download_as_bytes()
                        print(f"      (원격 GCS 백업 매핑 적용: gs://ai-multimodal-data/{blob_name})")
                except Exception:
                    pass

            if video_bytes:
                encoded_video = base64.b64encode(video_bytes).decode('utf-8')
                video_html = f'''
                <div style="position: relative; z-index: 9999; pointer-events: auto !important; display: inline-block;">
                    <video width="400" controls  style="position: relative; z-index: 9999; pointer-events: auto !important; display: block;">
                        <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                    </video>
                </div>
                '''
                display(HTML(video_html))
            else:
                print(f"대시보드 비디오 미디어 플레이어 기동 실패: [Errno 2] No such file or directory: '{path}'")
        except Exception as e:
            print(f"대시보드 비디오 미디어 플레이어 기동 실패: {e}")
    else:
        print("갤러리 보관함에 복원 가능한 비디오 파일이 없습니다.")

# 3. Widget Event Handlers Definition with Diagnostic Guards
def on_search_clicked(b):
    with output_area:
        clear_output(wait=True)
        try:
            q_type = query_type_dropdown.value
            alpha = alpha_slider.value

            if q_type == 'text':
                query = text_query_input.value
                print(f"텍스트 질의 매칭 중: '{query}' (가중 파라미터 Alpha: {alpha})")
                if 'combined_hybrid_search' in globals():
                    raw_results = combined_hybrid_search(query, alpha=alpha, top_k=30)
                    results = []
                    for res in raw_results:
                        item = res['item']
                        results.append({
                            'id': item['id'],
                            'score': res['score'],
                            'type': item['type'],
                            'path': item['path']
                        })
                else:
                    print("통합 하이브리드 검색 엔진(combined_hybrid_search)이 없습니다. 순수 밀집 검색(Dense) 모드로 회귀 동작합니다.")
                    results = find_similar_content(query, 'text', top_k=30)

            elif q_type in ['image', 'video', 'audio']:
                if upload_button.value:
                    uploaded_files = upload_button.value
                    
                    if isinstance(uploaded_files, (tuple, list)):
                        if len(uploaded_files) > 0:
                            uploaded_file = uploaded_files[0]
                            content = uploaded_file['content']
                            filename = uploaded_file['name']
                        else:
                            print("업로드된 파일이 없습니다.")
                            return
                    elif isinstance(uploaded_files, dict):
                        if len(uploaded_files) > 0:
                            filename = list(uploaded_files.keys())[0]
                            content = uploaded_files[filename]['content']
                        else:
                            print("업로드된 파일이 없습니다.")
                            return
                    else:
                        print(f"지원하지 않는 업로드 구조 타입: {type(uploaded_files)}")
                        return

                    data_dir = globals().get('DATA_DIR', '.')
                    filepath = os.path.join(data_dir, f"uploaded_{filename}")
                    with open(filepath, 'wb') as f:
                        f.write(content)
                    print(f"업로드한 사용자 리소스 파일을 쿼리로 주입하여 크로스 매칭 중: {filename}")

                    results = find_similar_content(filepath, q_type, top_k=30)
                else:
                    print(f"{q_type} 질의를 수행하려면 먼저 업로드 컴포넌트를 통해 파일을 업로드해 주세요.")
                    return

            display_media_results(results)
            
        except Exception as err:
            import traceback
            print(f"❌ 검색 실행 중 내부 오류 발생: {err}")
            traceback.print_exc()

def on_shuffle_clicked(b):
    if 'combined_registry' not in globals() or not combined_registry:
        print("레지스트리 목록 내에 복원된 임베딩 행렬이 존재하지 않습니다.")
        return

    # Filter choice pool to exclude large full-length video files to prevent OOM
    valid_shuffle_items = [item for item in combined_registry if item['type'] in ['image', 'video_chunk']]
    if not valid_shuffle_items:
        print("셔플 선택할 수 있는 유효한 미디어 아이템이 없습니다.")
        return
    random_item = random.choice(valid_shuffle_items)
    random_file = random_item['id']
    path = random_item['path']

    with output_area:
        clear_output(wait=True)
        print(f"임의 셔플 시드 선택 쿼리: {random_file}")

        try:
            media_bytes = None
            if path.startswith("gs://"):
                from google.cloud import storage
                storage_client = storage.Client()
                parts = path[5:].split('/', 1)
                b_name = parts[0]
                bl_name = parts[1]
                bucket = storage_client.bucket(b_name)
                blob = bucket.blob(bl_name)
                media_bytes = blob.download_as_bytes()
            else:
                if os.path.exists(path):
                    with open(path, 'rb') as f:
                        media_bytes = f.read()
                else:
                    # Local path resolution
                    if '_' in random_item['id']:
                        chunk_idx = random_item['id'].rsplit('_', 1)[1]
                    else:
                        chunk_idx = random_item['id']
                    local_fallback_path = f"local_chunks/chunk_{chunk_idx}.mp4"
                    if os.path.exists(local_fallback_path):
                        with open(local_fallback_path, 'rb') as f:
                            media_bytes = f.read()

            if media_bytes:
                if random_item['type'] == 'image':
                    display(IPImage(data=media_bytes, width=200))
                    results = find_similar_content(path, 'image', top_k=30)
                elif random_item['type'] in ['video', 'video_chunk']:
                    encoded_video = base64.b64encode(media_bytes).decode('utf-8')
                    video_html = f'''
                    <div style="position: relative; z-index: 9999; pointer-events: auto !important; display: inline-block;">
                        <video width="300" controls  style="position: relative; z-index: 9999; pointer-events: auto !important; display: block;">
                            <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                        </video>
                    </div>
                    '''
                    display(HTML(video_html))
                    results = find_similar_content(path, 'video', top_k=30)
            else:
                # Try re-shuffling if path does not exist
                on_shuffle_clicked(b)
                return

            display_media_results(results)

        except Exception as e:
            import traceback
            print(f"❌ 셔플 검색 처리 중 예외 발생: {e}")
            traceback.print_exc()

# 4. Render UI and Bind Listeners
display(widgets.VBox([
    query_type_dropdown,
    widgets.HBox([text_query_input, upload_button]),
    alpha_slider,
    widgets.HBox([search_button, shuffle_button]),
    output_area
]))

search_button.on_click(on_search_clicked)
shuffle_button.on_click(on_shuffle_clicked)
print("대시보드 UI 인터렉티브 콤보박스 및 스크롤 바 레이아웃 로드 완료.")

## 6단계: 자원 해제 및 청소 (Resource Cleanup)

실습이 끝난 후 불필요한 과금을 예방하기 위해 Vector Search 2.0 서버리스 컬렉션을 즉각적으로 삭제해주는 파괴적 API 명령어입니다.
해당 셀은 오작동이나 무의식적 일괄 재생(Run All)으로 인한 인덱스 삭제 실수를 방지하고자 노트북 상에서 실행 불가능한 **Raw** 타입(Raw NBConvert) 셀로 특수 전환되어 보호받고 있습니다.
